<a href="https://colab.research.google.com/github/jaumg2004/xGMobile/blob/main/projeto_xGMobile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import numpy as np
import pandas as pd
import re
import unicodedata
from matplotlib import pyplot as plt
from abc import ABC, abstractmethod
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Projeto final de Python-xGMobile



In [30]:
data = [
    {
        "company": "  Amazôn  ",
        "industry": "e-COMMERCE ",
        "country": " brasil ",
        "revenue_usd": " 453546400000 ",
        "ai_roi_percent": "30.33%",
        "ai_maturity_score": "77"
    },
    {
        "company": "OpenAI",
        "industry": " Technology",
        "country": "USA",
        "revenue_usd": "180000000",
        "ai_roi_percent": "19,5%",
        "ai_maturity_score": "101"
    },
    {
        "company": "  Microsoft ",
        "industry": "TECHNOLOGY",
        "country": " united states ",
        "revenue_usd": "211915000000",
        "ai_roi_percent": "28%",
        "ai_maturity_score": "95"
    },
    {
        "company": "Goógle  ",
        "industry": " technology ",
        "country": "USA ",
        "revenue_usd": "307394000000 ",
        "ai_roi_percent": "26,7%",
        "ai_maturity_score": "91"
    },
    {
        "company": "  Nubank",
        "industry": " FinTech ",
        "country": "Brasil",
        "revenue_usd": "8000000000",
        "ai_roi_percent": "18%",
        "ai_maturity_score": "84"
    },
    {
        "company": "Teslá ",
        "industry": " Automotive",
        "country": " usa",
        "revenue_usd": "96773000000",
        "ai_roi_percent": "22.4%",
        "ai_maturity_score": "89"
    },
    {
        "company": "  Samsung",
        "industry": "electronics ",
        "country": " south korea ",
        "revenue_usd": "200000000000",
        "ai_roi_percent": "17,2%",
        "ai_maturity_score": "82"
    },
    {
        "company": "Mercadó Livre",
        "industry": " E-commerce",
        "country": " argentina ",
        "revenue_usd": "14800000000 ",
        "ai_roi_percent": "24%",
        "ai_maturity_score": "86"
    },
    {
        "company": "  Petrobras ",
        "industry": "Energy ",
        "country": " BRASIL",
        "revenue_usd": "124474000000",
        "ai_roi_percent": "12,8%",
        "ai_maturity_score": "73"
    },
    {
        "company": "IBM",
        "industry": "Technology  ",
        "country": "usa",
        "revenue_usd": "61860000000",
        "ai_roi_percent": "15%",
        "ai_maturity_score": "88"
    },
    {
        "company": "  ifood",
        "industry": "food tech ",
        "country": "Brasil ",
        "revenue_usd": "2200000000",
        "ai_roi_percent": "21,1%",
        "ai_maturity_score": "79"
    },
    {
        "company": "AliBába",
        "industry": "E-COMMERCE",
        "country": " china ",
        "revenue_usd": "126491000000",
        "ai_roi_percent": "23%",
        "ai_maturity_score": "90"
    },
    {
        "company": "  Siemens ",
        "industry": "industrial automation",
        "country": " germany",
        "revenue_usd": "83000000000 ",
        "ai_roi_percent": "14.6%",
        "ai_maturity_score": "81"
    },
    {
        "company": "Spotify ",
        "industry": " MediaTech ",
        "country": " sweden ",
        "revenue_usd": "15000000000",
        "ai_roi_percent": "16%",
        "ai_maturity_score": "76"
    },
    {
        "company": "  ByteDance",
        "industry": "technology",
        "country": "China ",
        "revenue_usd": "110000000000",
        "ai_roi_percent": "27,3%",
        "ai_maturity_score": "93"
    },
    {
        "company": "Magalu",
        "industry": " retail ",
        "country": " brasil ",
        "revenue_usd": "35000000000",
        "ai_roi_percent": "11%",
        "ai_maturity_score": "72"
    }
]

In [31]:
class SanitizationError(Exception):
    """Erro base do processo de saneamento."""
    pass

class SchemaValidationError(SanitizationError):
    """Erro lançado quando um campo obrigatório está ausente ou inválido."""
    pass

In [32]:
class ScoreDescriptor:
    """Valida se um atributo numérico está entre 0 e 100."""

    def __set_name__(self, owner, name):
        self.private_name = "_" + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.private_name, None)

    def __set__(self, instance, value):
        if not (0 <= value <= 100):
            raise ValueError(f"{self.private_name} must be between 0 and 100.")
        setattr(instance, self.private_name, value)

In [33]:
class RecordModel:
    """Modelo simples para validar score com descritor."""
    ai_maturity_score = ScoreDescriptor()

    def __init__(self, ai_maturity_score):
        self.ai_maturity_score = ai_maturity_score

In [34]:
class BaseSanitizer(ABC):
    """Interface padrão dos saneadores do SASD."""

    def __init__(self, records):
        self.records = records

    @abstractmethod
    def calibrate(self):
        """Define regras de saneamento."""
        pass

    @abstractmethod
    def clean(self):
        """Aplica a limpeza aos dados."""
        pass

In [35]:
class TextSanitizer(BaseSanitizer):
    """Saneador de campos textuais."""

    def calibrate(self):
        self.text_fields = ["company", "industry", "country"]

    @staticmethod
    def normalize_text(value):
        value = str(value).strip().lower()
        value = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode("utf-8")
        return re.sub(r"\s+", " ", value)

    def clean(self):
        self.calibrate()
        cleaned = []
        for record in self.records:
            new_record = record.copy()
            for field in self.text_fields:
                if field in new_record:
                    new_record[field] = self.normalize_text(new_record[field])
            cleaned.append(new_record)
        return cleaned


In [36]:
class NumericSanitizer(TextSanitizer):
    """Saneador numérico especializado."""

    def calibrate(self):
        super().calibrate()
        self.numeric_fields = ["revenue_usd", "ai_roi_percent", "ai_maturity_score"]

    @staticmethod
    def parse_number(value):
        value = str(value).strip().replace("%", "").replace(",", ".")
        return float(value)

    @classmethod
    def required_fields(cls):
        return ["company", "industry", "country", "revenue_usd", "ai_roi_percent", "ai_maturity_score"]

    def clean(self):
        records = super().clean()
        self.calibrate()

        cleaned = []
        for record in records:
            new_record = record.copy()
            try:
                for field in self.numeric_fields:
                    new_record[field] = self.parse_number(new_record[field])

                model = RecordModel(new_record["ai_maturity_score"])
                new_record["ai_maturity_score"] = model.ai_maturity_score

                cleaned.append(new_record)
            except ValueError as exc:
                raise SchemaValidationError(f"Invalid numeric value in record: {record}") from exc

        return cleaned


In [37]:
class SchemaValidator(NumericSanitizer):
    """Valida a presença dos campos obrigatórios."""

    def clean(self):
        records = super().clean()
        required = self.required_fields()

        validated = []
        for record in records:
            missing = list(filter(lambda field: field not in record, required))
            if missing:
                raise SchemaValidationError(f"Missing fields: {missing}")
            validated.append(record)

        return validated

In [38]:
class SASDPipeline:
    """Executa o pipeline completo de saneamento."""

    def __init__(self, data):
        self.data = data

    def run(self):
        sanitizer = SchemaValidator(self.data)
        cleaned_records = sanitizer.clean()
        return pd.DataFrame(cleaned_records)

In [39]:
pipeline = SASDPipeline(data)
df_final = pipeline.run()
df_final

SchemaValidationError: Invalid numeric value in record: {'company': 'openai', 'industry': 'technology', 'country': 'usa', 'revenue_usd': '180000000', 'ai_roi_percent': '19,5%', 'ai_maturity_score': '101'}